# Reranking Methods — Interactive Notebook

This notebook provides an interactive environment for running and comparing **LLM-based** and **Cohere-based** reranking methods on STARK Knowledge Base (SKB) queries.

| Method | Name | Backend |
|--------|------|---------|
| 0 | LLM Score-Based | LlmBridge |
| 1 | LLM List-Sort | LlmBridge |
| 2 | LLM Pairwise | LlmBridge |
| 3 | Neural Reranking | Cohere rerank-v3.5 |

**Workflow:** Run sections 1→3 once to set up, then jump to the method section you want, then run section 8→10.


---
## Section 1 — Imports & Configuration

Import all libraries and define all tunable parameters here. **Edit this cell before running anything else.**

In [2]:
!pip install cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 8.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 10.7 MB/s eta 0:00:00a 0:00:01

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [35]:

import os
import ast
import time
import functools
import threading
import concurrent.futures
from collections import deque

import pandas as pd
import torch
import openai
import cohere
from tqdm.notebook import tqdm

from stark_qa import load_skb
from custom_pipeline.llm_bridge import LlmBridge

# ─────────────────────────────────────────────
# I/O Paths
# ─────────────────────────────────────────────
INPUT_CSV   = "full_data_dump.csv"          # Path to input CSV
OUTPUT_CSV  = "output/reranked_output.csv"  # Where results will be saved

# ─────────────────────────────────────────────
# Dataset & Method
# ─────────────────────────────────────────────
DATASET_NAME     = "prime"   # "prime" | "amazon" | "mag"
RERANKING_METHOD = 3         # 0=LLM Score | 1=LLM List-Sort | 2=LLM Pairwise | 3=Cohere

# 0 = use results["answer_list"]
# 1 = use "vss_merged_candidates" column
METRICS_ON = 0

# ─────────────────────────────────────────────
# LLM Settings  (Methods 0, 1, 2)
# ─────────────────────────────────────────────
MODEL_NAME   = "meta/llama-3.3-70b-instruct"
CONFIGS_PATH = "configs.json"

# ─────────────────────────────────────────────
# Reranking Hyper-parameters
# ─────────────────────────────────────────────
MAX_K       = 20
SIM_WEIGHT  = 0.1     # blending weight for positional similarity score (method 0)
COMPACT_DOCS = False  # use compact doc representation
ADD_REL     = True    # include relations in doc info

# ─────────────────────────────────────────────
# Cohere Settings  (Method 3)
# ─────────────────────────────────────────────
COHERE_API_KEYS = [
    "zUIi4u2suP7wMEOGZOeQaVP4LcXHvs7U1I4Avswd",
    "ki5sgOxJy797ufk7fhQiK7LE5l3DJiK9CTyGefQ2",
    "PS45TDnnGiQXwPafsH0SoN2w0IekBgjGLfJ9aiBJ",
    "NK3nihzYwIq73MmVEJavCGdUUvBsfoakrihCw0ef",
]
COHERE_RATE_LIMIT_PER_MIN = 10
MAX_WORKERS     = 40   # parallel threads for Cohere reranking (method 3)
MAX_WORKERS_LLM = 8    # parallel threads for LLM reranking   (methods 0, 1, 2)

print("✓ Imports and configuration loaded.")
print(f"  Dataset      : {DATASET_NAME}")
print(f"  Method       : {RERANKING_METHOD}")
print(f"  Input CSV    : {INPUT_CSV}")
print(f"  Output CSV   : {OUTPUT_CSV}")
print(f"  LLM workers  : {MAX_WORKERS_LLM}")


✓ Imports and configuration loaded.
  Dataset      : prime
  Method       : 3
  Input CSV    : full_data_dump.csv
  Output CSV   : output/reranked_output.csv
  LLM workers  : 8


---
## Section 2 — Data Loading & Preprocessing

Load the input CSV and extract the candidate answer lists for reranking.

In [27]:
df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df)} rows from '{INPUT_CSV}'")

# Parse serialised columns
df["results"] = df["results"].map(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

if METRICS_ON == 1:
    df["vss_merged_candidates"] = df["vss_merged_candidates"].map(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )
    df["answer_list"] = df["vss_merged_candidates"]
else:
    df["answer_list"] = [row["answer_list"] if "answer_list" in row else [] for row in df["results"]]

# Truncate candidates to MAX_K
df["answer_list"] = df["answer_list"].map(lambda x: x[:MAX_K])
# find zero length queries
print(f"After filtering zero-length candidates, {len(df)} rows remain.")

# Parse ground truths
if "ground_truths" in df.columns:
    df["ground_truths"] = df["ground_truths"].map(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

print(f"\nSample answer_list length : {len(df['answer_list'].iloc[0])}")
print(f"Columns available         : {list(df.columns)}\n")
display(df[["id", "query", "answer_list", "ground_truths"]].head())


Loaded 127 rows from 'full_data_dump.csv'
After filtering zero-length candidates, 127 rows remain.

Sample answer_list length : 20
Columns available         : ['id', 'query', 'ground_truths', 'status', 'entity_id_response', 'relations_id_response', 'entities', 'initial_symbol_candidates', 'relations', 'results', 'answer_list']



,id,query,answer_list,ground_truths
0,2614,What sports team logo pillowcases would pair w...,"[273164, 220628, 235047, 220657, 71062, 365011...","[220628, 235047]"
1,8291,Can you suggest a pair of comfortable shoes th...,"[1032747, 28610, 146398, 134131, 134126, 10491...","[202369, 527293, 862982]"
2,614,What's a recommended Coleman catalytic heater ...,"[379990, 8904, 26734, 281511, 348496, 9778, 93...","[8904, 379990, 348496, 26734]"
3,2235,Can you recommend a set of long sleeve t-shirt...,"[878640, 911995, 746233, 319068, 849323, 95429...","[129643, 339379, 176820, 524222]"
4,8903,Can you suggest a comfortable and health-promo...,"[935875, 595471, 660591, 405885, 790253, 36002...",[935875]


---
## Section 3 — Knowledge Base & Model Initialization

Load the SKB, set up the LLM bridge (methods 0–2), and initialise the Cohere key manager (method 3).  
Run all three sub-cells — only the relevant clients will be used downstream.

In [28]:
# ── 3a: STARK Knowledge Base ──────────────────────────────
kb = load_skb(DATASET_NAME, download_processed=True)
print(f"✓ Loaded SKB '{DATASET_NAME}'")


Loading from /home/coep/.cache/huggingface/hub/datasets--snap-stanford--stark/snapshots/88269e23e90587f99476c5dd74e235a0877e69be/skb/prime/processed!
✓ Loaded SKB 'prime'


In [17]:
# ── 3b: LLM Bridge (Methods 0, 1, 2) ─────────────────────
llm = LlmBridge(
    model_name=MODEL_NAME,
    configs_path=CONFIGS_PATH,
    verbose=False,
    dataset=DATASET_NAME

)
print(f"✓ LlmBridge initialised  — model: {MODEL_NAME}")


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
✓ LlmBridge initialised  — model: meta/llama-3.3-70b-instruct


In [18]:
# ── 3c: Cohere Rate-Limited Key Manager (Method 3) ────────
class RateLimitedKeyManager:
    """Round-robins across multiple API keys respecting a per-minute rate limit."""

    def __init__(self, api_keys, rate_limit_per_min=10):
        self.api_keys = api_keys
        self.rate_limit = rate_limit_per_min
        self.usage = {key: deque() for key in api_keys}
        self.lock = threading.Lock()

    def get_key(self):
        while True:
            with self.lock:
                current_time = time.time()
                min_wait_time = float("inf")

                for key in self.api_keys:
                    timestamps = self.usage[key]
                    while timestamps and timestamps[0] < current_time - 62:
                        timestamps.popleft()

                    if len(timestamps) < self.rate_limit:
                        timestamps.append(current_time)
                        return key

                    if timestamps:
                        wait_time = timestamps[0] + 62 - current_time
                        if wait_time < min_wait_time:
                            min_wait_time = wait_time

                if min_wait_time == float("inf"):
                    min_wait_time = 1.0

            time.sleep(min_wait_time)


key_manager = RateLimitedKeyManager(COHERE_API_KEYS, rate_limit_per_min=COHERE_RATE_LIMIT_PER_MIN)
print(f"✓ Cohere key manager initialised — {len(COHERE_API_KEYS)} API keys, "
      f"{COHERE_RATE_LIMIT_PER_MIN} req/min each")


✓ Cohere key manager initialised — 4 API keys, 10 req/min each


---
## Section 4 — Method 0: LLM Score-Based Reranking

Each candidate node is independently scored by the LLM (0.0 – 1.0). A positional similarity score is blended in and `torch.topk` is used to rank the results.

**Pros:** parallelisable via batch LLM call.  
**Cons:** relies on absolute score calibration.

In [29]:
def method0_reranking(top_k_node_ids, query, node_id_mask, kb, llm,
                      sim_weight, max_k, compact_docs, add_rel):
    cand_len = len(top_k_node_ids)
    pred_dict = {}

    prompts = []
    for idx, node_id in enumerate(top_k_node_ids):
        doc_info = kb.get_doc_info(node_id, add_rel=add_rel, compact=compact_docs)
        node_type = kb.get_node_type_by_id(node_id)
        prompts.append(
            f"Examine if a {node_type} satisfies a given query and assign a score from 0.0 to 1.0. "
            f"If the {node_type} does not satisfy the query, the score should be 0.0. "
            f"If there exists explicit and strong evidence supporting that {node_type} satisfies the query, "
            f"the score should be 1.0. If partial evidence or weak evidence exists, the score should be between 0.0 and 1.0.\n"
            f"Here is the query:\n\"{query}\"\n"
            f"Here is the information about the {node_type}:\n" +
            doc_info + "\n\n" +
            f"Please score the {node_type} based on how well it satisfies the query. "
            f"ONLY output the floating point score WITHOUT anything else. "
            f"Output: The numeric score of this {node_type} is: "
        )

    answers, _ = llm.ask_llm_batch(prompts, chat_logs=None)

    for idx, node_id in enumerate(top_k_node_ids):
        try:
            llm_score = float(answers[idx])
        except TypeError:
            if answers[idx] is None:
                if add_rel:
                    raise RuntimeError("None answer with add_rel=True")
                else:
                    llm_score = 0.5
        except ValueError:
            llm_score = 0.5

        sim_score = (cand_len - idx) / cand_len
        score = llm_score + sim_weight * sim_score

        if node_id_mask is not None:
            score /= 2
            if idx < len(node_id_mask):
                score += 0.5
        pred_dict[node_id] = score

    node_scores = torch.FloatTensor(list(pred_dict.values()))
    top_k_idx = torch.topk(
        node_scores,
        min(max_k, len(node_scores)),
        dim=-1, largest=True, sorted=True
    ).indices.tolist()
    return [list(pred_dict.keys())[i] for i in top_k_idx]


print("✓ method0_reranking defined")


✓ method0_reranking defined


---
## Section 5 — Method 1: LLM List-Sort Reranking

All candidates are listed in a single prompt and the LLM returns them sorted by relevance as comma-separated IDs.

**Pros:** single LLM call per query.  
**Cons:** LLM may hallucinate or omit IDs; output parsing can be fragile.

In [30]:
def method1_reranking(top_k_node_ids, query, node_id_mask, kb, llm, compact_docs):
    def _sort_list(node_ids_to_rerank, query):
        if not node_ids_to_rerank:
            return []

        possible_answers = "\n"
        for node_id in node_ids_to_rerank:
            doc_info = kb.get_doc_info(node_id, add_rel=False, compact=compact_docs)
            possible_answers += f"{node_id} {doc_info}\n"

        prompt = (
            f"The rows of the following list consist of an ID number, a type and a corresponding descriptive text:\n"
            f"{possible_answers}\n"
            f"Please sort this list in descending order according to how well the elements can be considered as "
            f"answers to the following query:\n{query}\n"
            f"Please make absolutely sure that the element which satisfies the query best is the first element. "
            f"Return ONLY the corresponding ID numbers separated by commas in the asked order."
        )

        output, _ = llm.ask_llm_batch([prompt], chat_logs=None)

        try:
            answer = [int(s.strip()) for s in output[0].split(",")]
            answer = list(dict.fromkeys(answer))
            sorted_ids = [n for n in answer if n in node_ids_to_rerank]
            print(f"  Invented IDs: {len(answer) - len(sorted_ids)}  |  "
                  f"Missing IDs: {len(node_ids_to_rerank) - len(sorted_ids)}")
        except Exception:
            sorted_ids = []
            print(f"  Could not parse LLM output: {output[0]}")

        # append any IDs the LLM omitted at the end
        sorted_ids += [n for n in node_ids_to_rerank if n not in sorted_ids]
        return sorted_ids

    return _sort_list(top_k_node_ids, query)


print("✓ method1_reranking defined")


✓ method1_reranking defined


---
## Section 6 — Method 2: LLM Pairwise Reranking

Every pair of candidates is compared by the LLM. Python's `sorted()` with `functools.cmp_to_key` drives the tournament.

**Pros:** very fine-grained ranking.  
**Cons:** $O(n \log n)$ LLM calls — expensive for large candidate lists.

In [31]:
def pairwise_comparison(node1_id, node2_id, query, kb, llm, compact_docs, add_rel):
    if node1_id == node2_id:
        return 0

    node_type_1 = kb.get_node_type_by_id(node1_id)
    node_type_2 = kb.get_node_type_by_id(node2_id)
    doc_info_1  = kb.get_doc_info(node1_id, add_rel=add_rel, compact=compact_docs)
    doc_info_2  = kb.get_doc_info(node2_id, add_rel=add_rel, compact=compact_docs)

    prompt = (
        f"The following two elements consist of an ID number, a type and a corresponding descriptive text:\n\n"
        f"{node1_id}, {node_type_1}, {doc_info_1}.\n"
        f"{node2_id}, {node_type_2}, {doc_info_2}.\n\n"
        f"Find out which of the elements satisfies the following query better:\n{query}\n"
        f"Return ONLY the corresponding ID number of the better element. Nothing else."
    )

    answer, _ = llm.ask_llm_batch([prompt], chat_logs=None)
    answer = answer[0]
    if isinstance(answer, str):
        answer = answer.replace("'", "").replace('"', "").strip()
    if answer == "A":
        answer = node1_id
    elif answer == "B":
        answer = node2_id

    try:
        answer = int(answer)
    except Exception:
        print(f"  Pairwise: cannot parse answer '{answer}'")
        return 0

    if answer == node1_id:
        return 1
    elif answer == node2_id:
        return -1
    else:
        print(f"  Pairwise: answer '{answer}' is neither {node1_id} nor {node2_id}")
        return 0


def method2_reranking(top_k_node_ids, query, node_id_mask, kb, llm, compact_docs, max_k):
    _cmp = functools.cmp_to_key(
        lambda a, b: pairwise_comparison(a, b, query=query, kb=kb, llm=llm,
                                         compact_docs=compact_docs, add_rel=True)
    )
    try:
        return sorted(top_k_node_ids, key=_cmp, reverse=True)
    except RuntimeError:
        _cmp_norel = functools.cmp_to_key(
            lambda a, b: pairwise_comparison(a, b, query=query, kb=kb, llm=llm,
                                             compact_docs=compact_docs, add_rel=False)
        )
        return sorted(top_k_node_ids, key=_cmp_norel, reverse=True)


print("✓ pairwise_comparison & method2_reranking defined")


✓ pairwise_comparison & method2_reranking defined


---
## Section 7 — Method 3 (v2): Cohere Neural Reranking

Uses Cohere's `rerank-v3.5` model to score all candidates in one API call. Multiple API keys are round-robined to stay within rate limits, parallelised across queries with `ThreadPoolExecutor`.

**Pros:** fast, neural relevance signal, no prompt engineering.  
**Cons:** requires Cohere API access; rate-limited.

In [36]:
def rerank_with_cohere_direct(query, documents, top_n=None):
    """Call Cohere rerank-v3.5 for a single query."""
    try:
        api_key = key_manager.get_key()
        co = cohere.ClientV2(api_key)
        response = co.rerank(
            model="rerank-v3.5",
            query=query,
            documents=documents,
            top_n=top_n,
            max_tokens_per_doc=48000,
        )
        return [{"index": r.index, "relevance_score": r.relevance_score}
                for r in response.results]
    except Exception as e:
        print(f"  Cohere API error: {e}")
        return []


def method3_rerank_single(top_k_node_ids, query, kb, max_k=20,
                          compact_docs=False, add_rel=False):
    top_k_node_ids = top_k_node_ids[:max_k]
    documents = [kb.get_doc_info(nid, add_rel=add_rel, compact=compact_docs)
                 for nid in top_k_node_ids]

    results = rerank_with_cohere_direct(query, documents, top_n=len(documents))
    if not results:
        print("  Cohere returned no results — keeping original order.")
        return top_k_node_ids
    return [top_k_node_ids[r["index"]] for r in results]


def process_single_query_with_retry(row, kb, max_k, compact_docs, add_rel, max_retries=3):
    query_text  = row["query"]
    answer_list = row["answer_list"]
    for attempt in range(max_retries):
        try:
            return method3_rerank_single(answer_list, query_text, kb,
                                         max_k=max_k, compact_docs=compact_docs,
                                         add_rel=add_rel)
        except Exception as e:
            print(f"  Attempt {attempt+1}/{max_retries} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(2 * (attempt + 1))
    return answer_list  # fallback


def rerank_queries_parallel(df_to_process, kb, max_k, compact_docs, add_rel, max_workers=40):
    print(f"Parallel Cohere reranking — {max_workers} workers …")
    futures = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        for _, row in df_to_process.iterrows():
            futures.append(executor.submit(
                process_single_query_with_retry, row, kb, max_k, compact_docs, add_rel
            ))
        for _ in tqdm(concurrent.futures.as_completed(futures), total=len(futures),
                      desc="Cohere reranking"):
            pass
    return [f.result() for f in futures]   # preserve original row order


print("✓ method3 / Cohere reranking helpers defined")


✓ method3 / Cohere reranking helpers defined


---
## Section 8 — Run Selected Reranking Method

Dispatches to the method chosen via `RERANKING_METHOD` in Section 1.  
Results are stored in `df_new`.

In [37]:

def _rerank_one_query(idx, top_k, query, reranking_method, kb, llm,
                      sim_weight, max_k, compact_docs, add_rel):
    """Rerank candidates for a single query. Returns (idx, result_list)."""
    if not top_k:
        return idx, []

    if reranking_method == 0:
        try:
            result = method0_reranking(top_k, query, None, kb, llm,
                                       sim_weight, max_k, compact_docs, add_rel=True)
        except (RuntimeError, openai.BadRequestError):
            result = method0_reranking(top_k, query, None, kb, llm,
                                       sim_weight, max_k, compact_docs, add_rel=False)

    elif reranking_method == 1:
        result = method1_reranking(top_k, query, None, kb, llm, compact_docs)

    elif reranking_method == 2:
        result = method2_reranking(top_k, query, None, kb, llm, compact_docs, max_k)

    else:
        raise NotImplementedError(f"Unknown reranking method: {reranking_method}")

    return idx, result


def run_reranking(df, reranking_method, kb, llm,
                  sim_weight, max_k, compact_docs, add_rel, max_workers,
                  max_workers_llm=1):

    # ── Filter answer lists to valid KB node IDs ──────────
    valid_ids = set(kb.get_candidate_ids())
    original_total = df["answer_list"].map(len).sum()

    df = df.copy()
    df["answer_list"] = df["answer_list"].map(
        lambda lst: [nid for nid in lst if nid in valid_ids]
    )

    filtered_total = df["answer_list"].map(len).sum()
    removed = original_total - filtered_total
    if removed > 0:
        print(f"⚠  Filtered out {removed} node IDs not present in '{DATASET_NAME}' SKB.")

    reranked_answers = [None] * len(df)

    if reranking_method == 3:
        # Cohere — parallel (progress bar is inside rerank_queries_parallel)
        reranked_answers = rerank_queries_parallel(
            df, kb=kb, max_k=max_k, compact_docs=compact_docs,
            add_rel=add_rel, max_workers=max_workers
        )

    else:
        method_names = {0: "LLM Score-Based", 1: "LLM List-Sort", 2: "LLM Pairwise"}
        desc = f"Method {reranking_method} · {method_names[reranking_method]}"

        print(f"Parallel LLM reranking — {max_workers_llm} worker(s) …")

        with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers_llm) as executor:
            # Submit all queries; keep a map future → row index for order preservation
            future_to_idx = {
                executor.submit(
                    _rerank_one_query,
                    i,
                    df["answer_list"].iloc[i][:max_k],
                    df["query"].iloc[i],
                    reranking_method, kb, llm,
                    sim_weight, max_k, compact_docs, add_rel,
                ): i
                for i in range(len(df))
            }

            completed = 0
            pbar = tqdm(
                concurrent.futures.as_completed(future_to_idx),
                total=len(df),
                desc=desc,
                unit="query",
                bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]{postfix}",
            )

            for future in pbar:
                idx, result = future.result()
                reranked_answers[idx] = result
                completed += 1
                query = df["query"].iloc[idx]
                pbar.set_postfix({
                    "done": completed,
                    "q": query[:40] + "…" if len(query) > 40 else query,
                })

    return reranked_answers


# ── Execute ───────────────────────────────────────────────
reranked_answers = run_reranking(
    df,
    reranking_method=RERANKING_METHOD,
    kb=kb,
    llm=llm,
    sim_weight=SIM_WEIGHT,
    max_k=MAX_K,
    compact_docs=COMPACT_DOCS,
    add_rel=ADD_REL,
    max_workers=MAX_WORKERS,
    max_workers_llm=MAX_WORKERS_LLM,
)

df_new = pd.DataFrame({
    "id":               df["id"].values,
    "ground_truths":    df["ground_truths"].values,
    "query":            df["query"].values,
    "original_answers": df["answer_list"].values,
    "reranked_answers": reranked_answers,
})

print(f"\n✓ Reranking complete — {len(df_new)} rows")
display(df_new.head())


⚠  Filtered out 1982 node IDs not present in 'prime' SKB.
Parallel Cohere reranking — 40 workers …


Cohere reranking:   0%|          | 0/127 [00:00<?, ?it/s]

  Cohere API error: headers: {'access-control-expose-headers': 'X-Debug-Trace-ID', 'cache-control': 'no-cache, no-store, no-transform, must-revalidate, private, max-age=0', 'content-encoding': 'gzip', 'content-type': 'application/json', 'expires': 'Thu, 01 Jan 1970 00:00:00 GMT', 'pragma': 'no-cache', 'vary': 'Origin,Accept-Encoding', 'x-accel-expires': '0', 'x-debug-trace-id': '06270b8b692a0b50cb4b872a7898f04e', 'x-endpoint-monthly-call-limit': '1000', 'x-trial-endpoint-call-limit': '40', 'x-trial-endpoint-call-remaining': '39', 'date': 'Sat, 07 Mar 2026 05:08:58 GMT', 'x-envoy-upstream-service-time': '19', 'server': 'envoy', 'via': '1.1 google', 'alt-svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000', 'transfer-encoding': 'chunked'}, status_code: 400, body: {'id': '7882acef-c0a8-4270-b139-265f68058237', 'message': 'invalid request: list of documents must not be empty'}
  Cohere returned no results — keeping original order.
  Cohere API error: headers: {'access-control-expose-head

KeyboardInterrupt: 

---
## Section 9 — Metrics Evaluation

Compute **Hit@1**, **Hit@5**, **MRR**, and **Recall@20** for the reranked results and display per-query + aggregate scores.

In [ ]:
def calculate_metrics(predicted_nodes, ground_truth_nodes):
    """Return Hit@1, Hit@5, MRR, Recall@20 for a single query."""
    if not predicted_nodes:
        return {"hit_at_1": 0.0, "hit_at_5": 0.0, "mrr": 0.0, "recall_at_20": 0.0}

    gt_set = set(ground_truth_nodes)

    hit_at_1 = 1.0 if predicted_nodes[0] in gt_set else 0.0
    hit_at_5 = 1.0 if any(n in gt_set for n in predicted_nodes[:5]) else 0.0

    mrr = 0.0
    for rank, node in enumerate(predicted_nodes, 1):
        if node in gt_set:
            mrr = 1.0 / rank
            break

    recall_at_20 = len(gt_set & set(predicted_nodes)) / len(gt_set) if gt_set else 0.0

    return {"hit_at_1": hit_at_1, "hit_at_5": hit_at_5,
            "mrr": mrr, "recall_at_20": recall_at_20}


def evaluate_dataframe_metrics(df):
    rows = []
    for i in range(len(df)):
        reranked = df["reranked_answers"].iloc[i]
        gt       = df["ground_truths"].iloc[i]
        if isinstance(reranked, str): reranked = ast.literal_eval(reranked)
        if isinstance(gt, str):       gt       = ast.literal_eval(gt)

        m = calculate_metrics(reranked, gt)
        m["id"]    = df["id"].iloc[i]
        m["query"] = df["query"].iloc[i]
        rows.append(m)

    metrics_df = pd.DataFrame(rows)

    print("=" * 45)
    print(f"  Method {RERANKING_METHOD} — {DATASET_NAME.upper()} — Aggregate Metrics")
    print("=" * 45)
    print(f"  Hit@1      : {metrics_df['hit_at_1'].mean():.4f}")
    print(f"  Hit@5      : {metrics_df['hit_at_5'].mean():.4f}")
    print(f"  MRR        : {metrics_df['mrr'].mean():.4f}")
    print(f"  Recall@20  : {metrics_df['recall_at_20'].mean():.4f}")
    print("=" * 45)
    return metrics_df


metrics_df = evaluate_dataframe_metrics(df_new)
display(metrics_df[["id", "query", "hit_at_1", "hit_at_5", "mrr", "recall_at_20"]].head(20))


---
## Section 10 — Save Results

Saves three files derived from `OUTPUT_CSV`:
- **`<name>.csv`** — full reranked results
- **`<name>_details.csv`** — per-query metric breakdown
- **`<name>_summary.csv`** — single-row aggregate stats

In [ ]:
os.makedirs(os.path.dirname(OUTPUT_CSV) or ".", exist_ok=True)

base_dir  = os.path.dirname(OUTPUT_CSV) or "."
base_name = os.path.splitext(os.path.basename(OUTPUT_CSV))[0]

# 1. Full reranked results
df_new.to_csv(OUTPUT_CSV, index=False)
print(f"✓ Reranked results   → {OUTPUT_CSV}")

# 2. Per-query metrics
details_path = os.path.join(base_dir, f"{base_name}_details.csv")
metrics_df.to_csv(details_path, index=False)
print(f"✓ Per-query metrics  → {details_path}")

# 3. Aggregate summary
summary_df = pd.DataFrame([{
    "Method":     RERANKING_METHOD,
    "Dataset":    DATASET_NAME,
    "Hit@1":      metrics_df["hit_at_1"].mean(),
    "Hit@5":      metrics_df["hit_at_5"].mean(),
    "MRR":        metrics_df["mrr"].mean(),
    "Recall@20":  metrics_df["recall_at_20"].mean(),
    "Num_queries": len(df_new),
}])
summary_path = os.path.join(base_dir, f"{base_name}_summary.csv")
summary_df.to_csv(summary_path, index=False)
print(f"✓ Aggregate summary  → {summary_path}")

display(summary_df)
